# Fundamentals of Data Science - Network Science Lab

# Lab Session 08 - The Configuration Model

In [ ]:
import igraph as ig
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path

In [ ]:
edges = [(0,1), (0,2), (1,2), (2,3), (3,4), (3,5), (4,5)]
g_py = ig.Graph(edges=edges, directed=False)

## The Configuration Model

The configuration model creates a random graph with a specified degree structure.

Under this setting, the probability that two nodes $i$ and $j$, which have degree $k_i$ and $k_j$ respectively, will be given by
$$p_{i, j} = \frac{k_ik_j}{2|E| - k_i}$$
For this reason, a configuration model allows to test whether a graph statistic can be explained by its degree distribution.

In general, a configuration model can be represented as an ensemble $G(n, \mathbf{k})$, where the degree sequence $\mathbf{k} = \{k_i\}_{i = 1}^n$ stores the degrees of each node in decreasing order.

### The Chung-Lu Model

Assuming the degree sequence only needs to resemble $\mathbf{k}$ in expectation, the Chung-Lu model allows to create a random graph using the same approach used for Erdős-Rényi networks.

For two nodes $i$ and $j$, which have degree $k_i$ and $k_j$ respectively, if $k_i = 1$, then the probability that node $i$ connects to node $j$ will be
$$p_{i, j} = \frac{k_j}{2m - 1} \text{, provided $\max_{i \in V} k_i^2 < 2m$ to ensure $p_{i, j} \leq 1$.}$$
For this reason, the resulting adjacency matrix will satisfy
$$A_{ij} = A_{ji} = \begin{cases}
  1 &\quad\text{with probability $p_{i, j}$} \\
  0 &\quad\text{otherwise} \\
\end{cases}$$
Overall, since the Chung-Lu model will resemble the degree distribution only in expectation, it is possible to observe that the degree of a specific node across several instances of the graph will follow a Poisson distribution with parameter $\lambda = k_i$.

Furthermore, while the Chung-Lu model is normally defined on undirected graphs, it can be extended to directed networks by specifying the in-degree sequence $\mathbf{k}_{\text{in}}$ and the out-degree sequence $\mathbf{k}_{\text{out}}$ and setting
$$p_{i, j} \approx \frac{k_i^{\text{out}}k_j^{\text{in}}}{m}$$

In [ ]:
random.seed(10) # For reproducibility.

# Create four instances of the Chung-Lu model.
chunglu1 = ig.Graph.Chung_Lu(g_py.degree(), loops = False)
chunglu2 = ig.Graph.Chung_Lu(g_py.degree(), loops = False)
chunglu3 = ig.Graph.Chung_Lu(g_py.degree(), loops = False)
chunglu4 = ig.Graph.Chung_Lu(g_py.degree(), loops = False)

# Plot the four graphs.
fig, ax = plt.subplots(2,2,figsize=(9, 8))

ig.plot(
    chunglu1,
    target=ax[0,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,0].set_title("First Graph")

ig.plot(
    chunglu2,
    target=ax[0,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,1].set_title("Second Graph")

ig.plot(
    chunglu3,
    target=ax[1,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,0].set_title("Third Graph")

ig.plot(
    chunglu4,
    target=ax[1,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,1].set_title("Fourth Graph")

plt.tight_layout()
plt.show()

### The Molloy-Reed Model

The Molloy-Reed model is a "stub matching" method because it represents a graph as a vector $\mathbf{v}$ of length $2|E|$ such that the index of node $i$ appears exactly $k_i$ times in the array.

Since each entry of the array represents a half-edge in the network, it is possible to consider a random permutation of the array to create a graph in which any adjacent pair or nodes is connected by an edge, eventually allowing self-loops or parallel edges.

In [ ]:
np.random.seed(11) # For reproducibility.

# Randomize the edge list of the bowtie graph to create four instances of the Molloy-Reed model.
g_py_edgelist = np.array(g_py.get_edgelist()).flatten()
molloyreed1 = ig.Graph(edges = np.random.choice(g_py_edgelist,
                                                len(g_py_edgelist), replace = False).reshape(-1, 2),
                       directed = False)
molloyreed2 = ig.Graph(edges = np.random.choice(g_py_edgelist,
                                                len(g_py_edgelist), replace = False).reshape(-1, 2),
                       directed = False)
molloyreed3 = ig.Graph(edges = np.random.choice(g_py_edgelist,
                                                len(g_py_edgelist), replace = False).reshape(-1, 2),
                       directed = False)
molloyreed4 = ig.Graph(edges = np.random.choice(g_py_edgelist,
                                                len(g_py_edgelist), replace = False).reshape(-1, 2),
                       directed = False)

# Plot the four graphs.
fig, ax = plt.subplots(2,2,figsize=(9, 8))

ig.plot(
    molloyreed1,
    target=ax[0,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,0].set_title("First Graph")

ig.plot(
    molloyreed2,
    target=ax[0,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,1].set_title("Second Graph")

ig.plot(
    molloyreed3,
    target=ax[1,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,0].set_title("Third Graph")

ig.plot(
    molloyreed4,
    target=ax[1,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,1].set_title("Fourth Graph")

plt.tight_layout()
plt.show()

### Monte Carlo Markov Chains

Monte Carlo Markov Chains aim to sample networks that follow a specified degree sequence but do not contain neither self-loops nor parallel edges.

The algorithm starts from a graph $G_0$ that has the target structural properties and the desired degree sequence $\mathbf{k}$ and repeatedly applies a simple function $g: G_t \to G_{t + 1}$ to create a sequence of graphs $G_0, G_1, \dots, G_t$ such that each term depends exclusively on the previous element in the sequence.

In general, a graph $G_t$ is used to create the next term $G_{t + 1}$ using the following procedure:
- Choose a uniformly random pair of edges $(u, v)$ and $(x, y)$.
- Create a new network $G_{t + 1}$ by replacing these edges with $(u, x)$ and $(v, y)$ or $(u, y)$ and $(v, x)$.
- If $G_{t + 1}$ does not satisfy the target structural properties, go back to $G_t$ and repeat the first two steps with another pair of edge.

Since this technique incrementally randomizes connections while preserving the desired degree sequence, it is possible to prove empirically that creating a network that is independent from the starting graph requires around $20|E|$ edge swaps, after which consecutive draws can be made about every $2|E|$ steps.

In [ ]:
np.random.seed(11) # For reproducibility.

# Create four instances of the Monte Carlo Markov Chain algorithm.
fosdick1 = g_py.copy()
fosdick1.rewire(n = 20*g_py.ecount(), mode = "simple")
fosdick2 = g_py.copy()
fosdick2.rewire(n = 20*g_py.ecount(), mode = "simple")
fosdick3 = g_py.copy()
fosdick3.rewire(n = 20*g_py.ecount(), mode = "simple")
fosdick4 = g_py.copy()
fosdick4.rewire(n = 20*g_py.ecount(), mode = "simple")

# Plot the four graphs.
fig, ax = plt.subplots(2,2,figsize=(9, 8))

ig.plot(
    fosdick1,
    target=ax[0,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,0].set_title("First Graph")

ig.plot(
    fosdick2,
    target=ax[0,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,1].set_title("Second Graph")

ig.plot(
    fosdick3,
    target=ax[1,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,0].set_title("Third Graph")

ig.plot(
    fosdick4,
    target=ax[1,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,1].set_title("Fourth Graph")

plt.tight_layout()
plt.show()

Note that the starting graph of a Monte Carlo Markov Chain algorithm typically comes from the Havel-Hakimi algorithm, which aims to determine whether a simple graph satisfies a given degree sequence.

Assuming the nodes of the graph are sorted in non-increasing order of degree in an array $L$, the algorithm recursively applies the following steps:
- Take the vertex $v_i$ with currently highest degree and connect it to the $i$ vertices immediately to its right within $L$, ultimately setting $k_i = 0$.
- Decrease the degree of the chosen nodes by $1$.
- Remove all the vertices that have degree $0$.

This procedure goes on until all nodes have degree $0$, which will require at most $|V| - 1$ steps for a degree sequence that can be represented by a simple graph.

In particular, if, at any step, one has $k_i > i - 1$, no simple graph satisfies the degree sequence because there are not enough nodes left for $i$ to connect to.